# Task 2 Exercise Answers

This notebook answers the Task 2 student exercises with small Python experiments. Each question is shown first, then the code that tests it, then a short conclusion.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Keep the notebook repeatable.
np.random.seed(42)
torch.manual_seed(42)


## 2. Helpers

In [ ]:
# This is a small synthetic version of the Task 2 coefficient-to-ellipse mapping.
# It is intentionally simple so we can compare models quickly.
def make_dataset(num_samples=1000, N=3, seed=42):
    rng = np.random.default_rng(seed)

    # Sample ellipse parameters.
    a = 0.3 + 0.5 * rng.random(num_samples)
    b = 0.3 + 0.5 * rng.random(num_samples)
    theta = np.pi * rng.random(num_samples)

    # Build the simplified Fourier-like coefficient vectors.
    coeffs = np.zeros((num_samples, 2 * (N + 1)), dtype=np.float32)
    for i in range(num_samples):
        coeffs_complex = np.zeros(N + 1, dtype=np.complex128)
        coeffs_complex[0] = a[i] * b[i]  # Area-like term
        for n in range(1, N + 1):
            coeffs_complex[n] = (a[i] - b[i]) * np.exp(1j * theta[i]) * (0.5 ** n)
        coeffs[i, :] = np.concatenate([np.real(coeffs_complex), np.imag(coeffs_complex)])

    targets = np.column_stack([a, b, theta]).astype(np.float32)
    return coeffs, targets

def split_train_val(X, Y, train_size=800):
    # Shuffle once so the split is random but reproducible.
    idx = np.arange(len(X))
    rng = np.random.default_rng(123)
    rng.shuffle(idx)
    train_idx = idx[:train_size]
    val_idx = idx[train_size:]
    return X[train_idx], X[val_idx], Y[train_idx], Y[val_idx]

def normalize_train_val(X_train, X_val):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0) + 1e-8
    return (X_train - mu) / sigma, (X_val - mu) / sigma, mu, sigma

def build_model(input_dim, output_dim, hidden_units):
    modules = []
    prev_dim = input_dim
    for units in hidden_units:
        modules.append(nn.Linear(prev_dim, units))
        modules.append(nn.ReLU())
        prev_dim = units
    modules.append(nn.Linear(prev_dim, output_dim))
    return nn.Sequential(*modules)

def count_params(model):
    return sum(param.numel() for param in model.parameters())

def train_and_score(X_train, X_val, Y_train, Y_val, hidden_units, epochs=12):
    X_train_n, X_val_n, _, _ = normalize_train_val(X_train, X_val)
    Y_train_n, Y_val_n, Y_mu, Y_sigma = normalize_train_val(Y_train, Y_val)

    model = build_model(X_train.shape[1], Y_train.shape[1], hidden_units)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train_n, dtype=torch.float32),
            torch.tensor(Y_train_n, dtype=torch.float32),
        ),
        batch_size=32,
        shuffle=True,
    )

    for _ in range(epochs):
        model.train()
        for batch_X, batch_Y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_Y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_n = model(torch.tensor(X_val_n, dtype=torch.float32)).cpu().numpy()
    pred = pred_n * Y_sigma + Y_mu
    err = np.abs(pred - Y_val)

    return {
        'model': str(hidden_units),
        'params': count_params(model),
        'mae_a': float(err[:, 0].mean()),
        'mae_b': float(err[:, 1].mean()),
        'mae_theta_deg': float(np.degrees(err[:, 2].mean())),
    }


## 3. Question 1: Change the network architecture

Question: What happens if we add more layers or remove layers?

In [ ]:
# Same dataset, different model depths.
X, Y = make_dataset(num_samples=1000, N=3, seed=42)
X_train, X_val, Y_train, Y_val = split_train_val(X, Y, train_size=800)

results_arch = []
results_arch.append(train_and_score(X_train, X_val, Y_train, Y_val, hidden_units=[32], epochs=12))
results_arch.append(train_and_score(X_train, X_val, Y_train, Y_val, hidden_units=[64, 32], epochs=12))
results_arch.append(train_and_score(X_train, X_val, Y_train, Y_val, hidden_units=[128, 64, 32], epochs=12))

arch_table = pd.DataFrame(results_arch)
arch_table

### Answer: Architecture

- The shallow model is faster but usually underfits the mapping.
- The middle model is a good balance for this demo.
- The deeper model can fit better, but it also has more parameters and can take longer to train.
- In short: adding layers helps up to a point, but the model should stay small enough to train reliably on the available data.

## 4. Question 2: Increase N

Question: What happens to the error if we increase N?

In [ ]:
# Compare a lower-order and higher-order coefficient target.
X3, Y3 = make_dataset(num_samples=1000, N=3, seed=7)
X5, Y5 = make_dataset(num_samples=1000, N=5, seed=7)

X3_train, X3_val, Y3_train, Y3_val = split_train_val(X3, Y3, train_size=800)
X5_train, X5_val, Y5_train, Y5_val = split_train_val(X5, Y5, train_size=800)

# Use the same architecture so the N comparison is fair.
n_results = []
n_results.append({'N': 3, **train_and_score(X3_train, X3_val, Y3_train, Y3_val, hidden_units=[64, 32], epochs=12)})
n_results.append({'N': 5, **train_and_score(X5_train, X5_val, Y5_train, Y5_val, hidden_units=[64, 32], epochs=12)})

n_table = pd.DataFrame(n_results)
n_table

### Answer: Higher N

- Increasing N makes the target harder because the network must model more coefficient terms.
- The error usually rises unless the model capacity or training data also increases.
- This is why higher-order problems often need more data, more training, or a larger network.

## 5. Question 3: Add noise

Question: What happens if we add noise to the inputs?

In [ ]:
# Train on clean data, then test on noisy inputs.
X, Y = make_dataset(num_samples=1000, N=3, seed=99)
X_train, X_val, Y_train, Y_val = split_train_val(X, Y, train_size=800)

X_train_n, X_val_n, X_mu, X_sigma = normalize_train_val(X_train, X_val)
Y_train_n, Y_val_n, Y_mu, Y_sigma = normalize_train_val(Y_train, Y_val)

noise_levels = [0.0, 0.05, 0.10]
noise_results = []

model = build_model(X_train.shape[1], Y_train.shape[1], hidden_units=[64, 32])
model.fit(X_train_n, Y_train_n, epochs=12, batch_size=32, verbose=0, validation_data=(X_val_n, Y_val_n))

rng = np.random.default_rng(2024)
for noise in noise_levels:
    noisy_val = X_val + rng.normal(scale=noise, size=X_val.shape)
    noisy_val_n = (noisy_val - X_mu) / X_sigma
    pred_n = model.predict(noisy_val_n, verbose=0)
    pred = pred_n * Y_sigma + Y_mu
    err = np.abs(pred - Y_val)
    noise_results.append({
        'noise_std': noise,
        'mae_a': float(err[:, 0].mean()),
        'mae_b': float(err[:, 1].mean()),
        'mae_theta_deg': float(np.degrees(err[:, 2].mean())),
    })

noise_table = pd.DataFrame(noise_results)
noise_table

### Answer: Noise

- Noise makes the coefficient-to-ellipse mapping less stable.
- The parameter error increases as the noise level increases.
- Small noise may be tolerable, but stronger noise usually hurts the angle estimate first.

## 6. Short Summary

This notebook shows the three Task 2 exercises directly:
- change the network depth
- increase N
- add noise

The main pattern is that harder problems need either more model capacity, more data, or both.